In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if Path('/kaggle/input').exists():
    repo_dir = Path('/kaggle/working/repo')
    if (repo_dir / '.git').exists():
        print('Kaggle repository exists. Updating main...')
        subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
    else:
        print('Kaggle environment detected. Cloning main...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/dvydinh/smpPrediction.git', str(repo_dir)], check=True)
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
    print(f'Repository commit {commit}')

os.system(f'{sys.executable} -m pip install -q -r requirements.txt')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import get_data_paths, load_and_preprocess_data
from src.feature_engineering import add_engineered_features
from src.feature_policy import select_production_features
from src import model_utils
from src import evaluation

DATA_ROOT, OUTPUT_DIR = get_data_paths()
print(f'Data root: {DATA_ROOT}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
df = load_and_preprocess_data(DATA_ROOT)

In [ ]:
df = add_engineered_features(df)

In [ ]:
feature_cols = select_production_features(df)
print(f'Feature count: {len(feature_cols)}')

model, selected_features = model_utils.train_and_save_model(
    df, feature_cols,
    output_dir=str(OUTPUT_DIR / 'models'),
)

In [ ]:
evaluation.evaluate_and_plot(
    model, df, selected_features,
    output_dir=str(OUTPUT_DIR / 'eval')
)